In [0]:
%pip install azure-eventhub requests sseclient-py

In [0]:
import json
import requests
from sseclient import SSEClient
from azure.eventhub import EventHubProducerClient, EventData


dbutils.widgets.text("secret_scope", "default2") 
dbutils.widgets.text("secret_key", "janvander0912-evh-cs") 
dbutils.widgets.text("evh_name", "janvander0912-evh")      

secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
evh_name = dbutils.widgets.get("evh_name")

print(f"Downloading Connection String from the vault: scope='{secret_scope}', key='{secret_key}'...")
try:
    conn_string = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    print(" -> Credentials retrieved successfully")
except Exception as e:
    print(f"ERROR: Failed to get secret '{secret_key}'. Make sure that safe '{secret_scope}' is correct.")
    raise e

In [0]:
def run_wikipedia_stream(max_events=150):
    print(f"Connecting to your Azure Event Hub ('{evh_name}') and Wikipedia API...")
    
    producer = EventHubProducerClient.from_connection_string(
        conn_str=conn_string,
        eventhub_name=evh_name
    )
    
    url = "https://stream.wikimedia.org/v2/stream/recentchange"
    headers = {"User-Agent": "Databricks-DataEngineer-Lab/1.0 (janvander0912)"}
    
    response = requests.get(url, stream=True, headers=headers)
    client = SSEClient(response)
    
    events_sent = 0
    
    with producer:
        # We create the first empty batch
        event_data_batch = producer.create_batch()
        
        for event in client.events():
            if event.event == "message":
                try:
                    raw_data = json.loads(event.data)
                    
                    # We extract the most interesting information about the edition and enrich the scheme
                    enriched_event = {
                        "event_id": events_sent + 1,
                        "user": raw_data.get("user"),
                        "article_title": raw_data.get("title"),
                        "wiki_domain": raw_data.get("server_url"),
                        "is_bot": raw_data.get("bot", False),
                        "length_change": raw_data.get("length", {}).get("new", 0) - raw_data.get("length", {}).get("old", 0),
                        "event_timestamp": raw_data.get("meta", {}).get("dt")
                    }
                    
                    payload = json.dumps(enriched_event)
                    
                    # We are trying to add an event to the current memory package
                    try:
                        event_data_batch.add(EventData(payload))
                        events_sent += 1
                    except ValueError:
                        # The batch has reached its byte size limit! We're sending it to the cloud...
                        producer.send_batch(event_data_batch)
                        print(f" -> Wysłano pełną paczkę w chmurę... (Razem: {events_sent} zdarzeń)")

                        event_data_batch = producer.create_batch()
                        event_data_batch.add(EventData(payload))
                        events_sent += 1
                        
                    if events_sent >= max_events:
                        # Last batch
                        producer.send_batch(event_data_batch)
                        print(f"\nSuccess! Exactly {events_sent} Wikipedia edits sent to your Event Hub.")
                        break
                        
                except Exception as parse_err:
                    pass

run_wikipedia_stream(max_events=150)